In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType,TimestampType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit


In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproysmartdata01")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/MessageTrace_2026.csv"

In [0]:
email_schema = StructType([
    StructField("Received", TimestampType(), True),
    StructField("SenderAddress", StringType(), True),
    StructField("RecipientAddress", StringType(), True),
    StructField("Subject", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("MessageTraceId", StringType(), True)
])

In [0]:
df_email = spark.read\
.option('header', True)\
.schema(email_schema)\
.csv(ruta)

In [0]:
# Normalizar nombres de columnas 
df_email = df_email.toDF(*[col.strip().lower() for col in df_email.columns])
df_email_final_df = df_email.select(
    col("received"),
    col("senderaddress"),
    col("recipientaddress"),
    col("subject"),
    col("status"),
    col("messagetraceid")
).withColumn("ingestion_date", current_timestamp())


In [0]:
df_email_final_df.write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.emails")

In [0]:
%sql
SELECT * FROM catalog_au.bronze.emails